# Phase 2/3 retraining (Kaggle)

Thin harness around the `retraining-phase23` repo. All logic lives in
`retrain/` modules; this notebook only wires Kaggle paths, secrets, and
the `cli.py` command.

Flow:
1. Install deps
2. Clone/pull the repo
3. Set config (phase, hyperparams, secrets via env — never hardcoded)
4. Validate + train (`cli.py train` fails loudly on bad data)
5. Eval (`cli.py eval`)

In [ ]:
# 1) Dependencies
%pip install -q transformers accelerate datasets peft bitsandbytes safetensors sentencepiece huggingface_hub neo4j pandas pyarrow tqdm

In [ ]:
# 2) Repo (clone once, pull on re-runs)
import os
REPO_URL = os.environ.get('RETRAIN_REPO', 'https://github.com/IAteNoodles/retraining-phase23.git')
REPO_DIR = '/kaggle/working/retraining-phase23'
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
print('repo at:', REPO_DIR)

In [ ]:
# 3) Config - edit these
import getpass, os
from pathlib import Path

PHASE = 2  # 2 or 3
DATA_DIR = Path('/kaggle/input/nl-cypher-lossless-phases')  # Kaggle dataset path
MODEL_DIR = Path('/kaggle/input/<your-model-dataset>')  # base model dir
OUTPUT_DIR = Path(f'/kaggle/working/phase{PHASE}_output')

SCHEMA_COL = 'schema_full'   # lossless column
SCHEMA_VARIANT = ''            # e.g. 'small,mid,full' on augmented files, '' = all

EPOCHS, LR, BATCH_SIZE, GRAD_ACCUM = 3, 1e-4, 4, 8
MAX_INPUT_LEN, MAX_TARGET_LEN, LORA_R = 1024, 320, 64
SEED = 42

# RL (needs Neo4j reachable from Kaggle: funnel URL works)
USE_RL = False
NEO4J_URL = 'https://pharmagaurd.tail5e1300.ts.net/db/{database}/query/v2'
NEO4J_USER = 'training'
NEO4J_PASSWORD = os.environ.get('NEO4J_PASSWORD', '') or getpass.getpass('Neo4j password: ')
DB_MAPPING = str(DATA_DIR / 'db_name_mapping.json')

# Hub sync (optional; leave empty to skip)
HF_REPO_ID = os.environ.get('HF_REPO_ID', '')
HF_TOKEN = os.environ.get('HF_TOKEN', '') or (getpass.getpass('HF token (blank to skip): ') or '')
RESUME = ''  # local path or hf://repo/path, '' = fresh

print('phase:', PHASE, '| data:', DATA_DIR.exists(), '| model:', MODEL_DIR.exists())

In [ ]:
# 4) Train (validates first, fails loudly on bad data)
import subprocess, sys

cmd = [sys.executable, f'{REPO_DIR}/cli.py', 'train',
    '--phase', str(PHASE),
    '--train_parquet', str(DATA_DIR / f'phase{PHASE}_train.parquet'),
    '--val_parquet', str(DATA_DIR / f'phase{PHASE}_val.parquet'),
    '--test_parquet', str(DATA_DIR / f'phase{PHASE}_test.parquet'),
    '--schema_col', SCHEMA_COL, '--question_col', 'question', '--target_col', 'target',
    '--model_path', str(MODEL_DIR), '--output_dir', str(OUTPUT_DIR),
    '--epochs', str(EPOCHS), '--lr', str(LR), '--batch_size', str(BATCH_SIZE),
    '--grad_accum', str(GRAD_ACCUM), '--max_input_length', str(MAX_INPUT_LEN),
    '--max_target_length', str(MAX_TARGET_LEN), '--lora_r', str(LORA_R),
    '--seed', str(SEED)]
if SCHEMA_VARIANT:
    cmd += ['--schema_variant', SCHEMA_VARIANT]
if USE_RL:
    cmd += ['--use_rl', '--rl_candidates', '4',
            '--neo4j_url', NEO4J_URL, '--neo4j_user', NEO4J_USER,
            '--neo4j_password', NEO4J_PASSWORD, '--db_mapping_json', DB_MAPPING]
if HF_REPO_ID:
    cmd += ['--hf_repo_id', HF_REPO_ID, '--hf_token', HF_TOKEN,
            '--upload_checkpoints', '--hf_path_prefix', f'phase_{PHASE}']
if RESUME:
    cmd += ['--resume_checkpoint', RESUME]
print(' '.join(cmd[:6]), '...')
ret = subprocess.run(cmd, cwd=REPO_DIR)
print('train exit:', ret.returncode)

In [ ]:
# 5) Eval the best adapter (uses the same shared eval module)
import subprocess, sys
cmd = [sys.executable, f'{REPO_DIR}/cli.py', 'eval',
    '--phase', str(PHASE),
    '--val_parquet', str(DATA_DIR / f'phase{PHASE}_val.parquet'),
    '--test_parquet', str(DATA_DIR / f'phase{PHASE}_test.parquet'),
    '--schema_col', SCHEMA_COL, '--question_col', 'question', '--target_col', 'target',
    '--model_path', str(MODEL_DIR),
    '--eval_model_dir', str(OUTPUT_DIR / 'best'),
    '--neo4j_url', NEO4J_URL, '--neo4j_user', NEO4J_USER,
    '--neo4j_password', NEO4J_PASSWORD, '--db_mapping_json', DB_MAPPING]
ret = subprocess.run(cmd, cwd=REPO_DIR)
print('eval exit:', ret.returncode)